# Heston Analytical Formula

### Heston

##### 
$$dS_t = rS_t\,dt + \sqrt{v_t}\,S_t\,dW_t^{(1)}$$
$$dv_t = \kappa(\theta - v_t)\,dt + \xi\sqrt{v_t}\,dW_t^{(2)}$$
$$dW_t^{(1)}\,dW_t^{(2)} = \rho\,dt$$

In [ ]:
import numpy as np
import scipy.integrate as spi
import matplotlib.pyplot as plt


### Semi-Analytical Heston Math (Characteristic Functions)


In [ ]:
def hs(phi, S0, v0, kappa, theta, xi, rho, r, tau, j):
    """
    Computes the characteristic function for Heston model probabilities P1 and P2.
    """
    if j == 1:
        u = 0.5
        b = kappa - rho * xi
    else:
        u = -0.5
        b = kappa

    a = kappa * theta
    x = np.log(S0)
    
    # Complex integration variables
    d = np.sqrt((rho * xi * phi * 1j - b)**2 - xi**2 * (2 * u * phi * 1j - phi**2))
    g = (b - rho * xi * phi * 1j + d) / (b - rho * xi * phi * 1j - d)
    
    # C and D equations
    C = r * phi * 1j * tau + a / xi**2 * ((b - rho * xi * phi * 1j + d) * tau - 2 * np.log((1 - g * np.exp(d * tau)) / (1 - g)))
    D = (b - rho * xi * phi * 1j + d) / xi**2 * ((1 - np.exp(d * tau)) / (1 - g * np.exp(d * tau)))
    
    return np.exp(C + D * v0 + 1j * phi * x)

def heston_integrand(phi, S0, v0, kappa, theta, xi, rho, r, tau, K, j):
    """
    The integrand to retrieve probabilities P1 and P2.
    """
    # Numerator of the integrand
    numerator = np.exp(-1j * phi * np.log(K)) * heston_char_func(phi, S0, v0, kappa, theta, xi, rho, r, tau, j)
    # Divided by (i * phi) and taking the real part
    return (numerator / (1j * phi)).real

def hs_scalar(S0, K, tau, r, v0, kappa, theta, xi, rho, type="call"):
    """
    Calculates the semi-analytical Heston price for a single scalar point.
    """
    # Handling option maturity (tau = 0 causes a singularity, so we just use the payoff formula)
    if tau <= 1e-4:
        if type == "call":
            return max(S0 - K, 0.0)
        else:
            return max(K - S0, 0.0)

    # Calculate P1 via numerical integration (scipy.integrate.quad)
    int1, _ = spi.quad(heston_integrand, 1e-4, 100, args=(S0, v0, kappa, theta, xi, rho, r, tau, K, 1))
    P1 = 0.5 + (1 / np.pi) * int1
    
    # Calculate P2 via numerical integration
    int2, _ = spi.quad(heston_integrand, 1e-4, 100, args=(S0, v0, kappa, theta, xi, rho, r, tau, K, 2))
    P2 = 0.5 + (1 / np.pi) * int2
    
    # Exact Call Price Formula
    call_price = S0 * P1 - K * np.exp(-r * tau) * P2
    
    if type == "call":
        return call_price
    else:
        # Put Option via Put-Call Parity
        return call_price - S0 + K * np.exp(-r * tau)

def hs(S, K, T, r, v0, kappa, theta, xi, rho, type="call"):
    """
    Vectorized version of the Heston analytical pricer to support numpy 2D grids (like meshgrid).
    Note: T represents tau (time to maturity) in this context.
    """
    # Numpy's vectorize allows our scalar function to execute over multidimensional grid matrices efficiently
    v_func = np.vectorize(hs_scalar)
    return v_func(S, K, T, r, v0, kappa, theta, xi, rho, type)


### Visualizing the Analytical Surface


In [ ]:
# Create a test grid to verify the pricing is working optimally
S_plot = np.linspace(1e-5, 300.0, 50)
tau_plot = np.linspace(0.01, 1.0, 50)
S_grid, tau_grid = np.meshgrid(S_plot, tau_plot)

# Calculate Heston Call prices analytically
analytical_surface = hs(
    S=S_grid, K=100.0, T=tau_grid, r=0.05, 
    v0=0.04, kappa=2.0, theta=0.04, xi=0.3, rho=-0.7, 
    type="call"
)

# Render the 3D surface plot
fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(111, projection='3d')
surf = ax.plot_surface(S_grid, tau_grid, analytical_surface, cmap=plt.cm.plasma,
                       linewidth=0, antialiased=False, alpha=0.9)

ax.set_xlabel('Stock Price (S)')
ax.set_ylabel('Time to Maturity (tau)')
ax.set_zlabel('Option Price (V)')
ax.set_title('Semi-Analytical Heston Call Option Surface')
fig.colorbar(surf, ax=ax, shrink=0.5, aspect=5)
plt.show()
